# Processamento de um lote PNCP com Docling

Use este notebook para um ZIP criado por `pipeline_lotes_documentos.py`. O estado de extração é isolado por período e lote.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import subprocess
import sys
from pathlib import Path

RAIZ_DRIVE = Path('/content/drive/MyDrive/pncp_rag')
PROJETO_DRIVE = RAIZ_DRIVE / 'projeto'
if not PROJETO_DRIVE.is_dir():
    raise FileNotFoundError(f'Projeto nao encontrado: {PROJETO_DRIVE}')

sys.path.insert(0, str(PROJETO_DRIVE))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJETO_DRIVE / 'requirements-colab.txt')], check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU indisponivel. Selecione T4 GPU no menu Runtime.')
print('GPU em uso:', torch.cuda.get_device_name(0))


In [ ]:
# ---------------- CONFIGURACAO DO LOTE ----------------
ANO = 2026
MES = 8
NOME_LOTE = 'lote_001'
LIMITE_DOCUMENTOS = 100     # execute 100 por vez dentro do ZIP do lote
TAMANHO_MICROLOTE = 50
PERFIL = 'qualidade'
REPROCESSAR_ERROS = False

ARQUIVO_ZIP_DRIVE = RAIZ_DRIVE / 'lotes' / f'{ANO}_{MES:02d}' / f'documentos_{NOME_LOTE}.zip'
RAIZ_DOCUMENTOS_COLAB = Path('/content/pncp_documentos')
if not ARQUIVO_ZIP_DRIVE.is_file():
    raise FileNotFoundError(f'ZIP do lote nao encontrado: {ARQUIVO_ZIP_DRIVE}')


In [ ]:
import shutil
import zipfile

ZIP_LOCAL = Path('/content/documentos.zip')
shutil.copy2(ARQUIVO_ZIP_DRIVE, ZIP_LOCAL)
shutil.rmtree(RAIZ_DOCUMENTOS_COLAB, ignore_errors=True)
RAIZ_DOCUMENTOS_COLAB.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_LOCAL) as arquivo_zip:
    arquivo_zip.extractall(RAIZ_DOCUMENTOS_COLAB)
print('Documentos prontos em:', RAIZ_DOCUMENTOS_COLAB)


In [ ]:
from src.base_estruturada.pipelines.pipeline_documentos import PeriodoColeta, caminhos_do_periodo
from src.base_estruturada.pipelines.pipeline_lotes_documentos import (
    caminhos_do_lote, carregar_lote, gerar_fila_docling_do_lote,
)
from src.base_estruturada.pipelines.pipeline_docling_lotes import executar_docling_por_lotes

periodo = PeriodoColeta(ano=ANO, mes=MES)
caminhos_periodo = caminhos_do_periodo(periodo)
caminhos_lote = caminhos_do_lote(periodo, NOME_LOTE)
lote = carregar_lote(PROJETO_DRIVE / caminhos_lote['lote'])
fila = gerar_fila_docling_do_lote(lote, PROJETO_DRIVE / caminhos_lote['fila_docling'])
print('PDFs na fila do lote:', fila['quantidadeDocumentos'])


In [ ]:
resumo = executar_docling_por_lotes(
    fila=fila,
    caminho_estado=PROJETO_DRIVE / caminhos_lote['estado_docling'],
    caminho_relatorio=PROJETO_DRIVE / caminhos_lote['relatorio_docling'],
    raiz_documentos=RAIZ_DOCUMENTOS_COLAB,
    pasta_saida=PROJETO_DRIVE / caminhos_periodo['pasta_textos_docling'],
    raiz_projeto_portavel=PROJETO_DRIVE,
    tamanho_microlote=TAMANHO_MICROLOTE,
    limite_documentos=LIMITE_DOCUMENTOS,
    perfil=PERFIL,
    reprocessar_erros=REPROCESSAR_ERROS,
)
print('Resumo final:', resumo)
